# Job Posting Analyzer
**Using the Gemini API to turn unstructured job postings into structured data, then analyzing which skills employers ask for.**

Pipeline:

1. **Raw text**: 20 Data Analyst job postings saved as `.txt` files in `postings/`
2. **LLM extraction**: each posting is sent to Gemini with a Pydantic schema; structured output guarantees the reply matches that schema
3. **Structured table**: results are collected into a pandas DataFrame and cached to `data/extracted.json`
4. **Analysis**: skill frequency, seniority, work mode, experience, salary, and a personal skill-gap check
5. **Validation**: spot-checking the model's output against the original postings

## 1. Setup
Loads the API key from `.env` and creates the Gemini client. Run cells top to bottom.

In [1]:
import os, json, time, random
from pathlib import Path
from typing import Literal, Optional

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv(override=True)
assert os.getenv("GEMINI_API_KEY"), "GEMINI_API_KEY not found. Create a .env file in this folder (see .env.example)."

# "gemini-flash-latest" always points to Google's current Flash model.
# To pin a specific version for reproducibility, replace it with a model name from https://ai.google.dev/gemini-api/docs/models
MODEL = "gemini-3.8-flash"
client = genai.Client()  # reads GEMINI_API_KEY from the environment

POSTINGS_DIR = Path("postings")
CACHE_FILE = Path("data/extracted.json")
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)
print(f"Model: {MODEL} | postings found: {len(list(POSTINGS_DIR.glob('*.txt')))}")

Model: gemini-3.8-flash | postings found: 20


## 2. Define the output schema
This Pydantic class is the contract with the model. `Literal` forces a choice from fixed categories, and `Optional` lets the model return `null` instead of inventing values that aren't in the posting (salary especially).

In [2]:
class JobPosting(BaseModel):
    job_title: str
    company: str
    seniority: Literal["intern", "entry", "mid", "senior", "lead"] = Field(
        description="Infer from title and years of experience: 0-1 yrs entry, 2-4 mid, 5+ senior.")
    work_mode: Literal["remote", "hybrid", "onsite", "unspecified"]
    min_years_experience: Optional[int] = Field(
        description="Minimum years of experience required; null if not stated.")
    required_skills: list[str] = Field(
        description="Technical skills and tools the posting lists as required.")
    nice_to_have_skills: list[str] = Field(
        description="Skills and tools listed as preferred, a plus, or nice to have.")
    salary_min: Optional[float] = Field(description="Lower bound of pay as stated; null if not stated.")
    salary_max: Optional[float] = Field(description="Upper bound of pay as stated; null if not stated.")
    pay_period: Literal["year", "hour", "unspecified"] = Field(
        description="Whether salary_min/max are annual or hourly; 'unspecified' if no pay is listed.")

## 3. Extraction function
The system instruction asks for **canonical skill names**. Without it, "SQL", "sql" and "Structured Query Language" would be counted as three different skills. Temperature 0 makes results as repeatable as possible. The function retries with a back-off if the API is rate-limited.

In [3]:
SYSTEM_PROMPT = """You extract structured information from job postings.
Rules:
- Skills are technical skills, tools, platforms or methods (not soft skills like "communication").
- Use short canonical names: "SQL", "Python", "R", "Excel", "Google Sheets", "Tableau", "Power BI",
  "Looker", "Snowflake", "dbt", "BigQuery", "Redshift", "Databricks", "Statistics", "A/B Testing",
  "Machine Learning", "Git", "AWS", "Data Modeling". Do not include version numbers or extra words.
- If a posting says "X or Y" (e.g. "Tableau or Power BI"), list each tool separately.
- Only put a skill in nice_to_have_skills if the posting marks it preferred / a plus / nice to have.
- Use null for anything not stated. Never guess salary or years of experience."""

def extract(posting_text: str, max_retries: int = 4) -> JobPosting:
    config = types.GenerateContentConfig(
        system_instruction=SYSTEM_PROMPT,
        response_mime_type="application/json",
        response_json_schema=JobPosting.model_json_schema(),
        temperature=0,
    )
    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(model=MODEL, contents=posting_text, config=config)
            return JobPosting.model_validate_json(response.text)
        except Exception as e:
            if attempt == max_retries - 1:
                raise
            wait = 2 ** attempt * 5
            print(f"  attempt {attempt + 1} failed ({type(e).__name__}: {str(e)[:80]}). Retrying in {wait}s...")
            time.sleep(wait)

## 4. Test on one posting
Compare this output with `postings/posting_01.txt` before running all twenty. If something is wrong, adjust `SYSTEM_PROMPT` above and re-run.

In [4]:
sample = Path("postings/posting_01.txt").read_text()
print(extract(sample).model_dump_json(indent=2))

  attempt 1 failed (ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp). Retrying in 5s...
  attempt 2 failed (ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp). Retrying in 10s...
{
  "job_title": "Data Analyst",
  "company": "LTK",
  "seniority": "mid",
  "work_mode": "remote",
  "min_years_experience": 2,
  "required_skills": [
    "SQL",
    "Looker",
    "Hex",
    "Wisdom",
    "Tableau",
    "Excel",
    "Google Sheets",
    "A/B Testing"
  ],
  "nice_to_have_skills": [
    "Python",
    "R"
  ],
  "salary_min": null,
  "salary_max": null,
  "pay_period": "unspecified"
}


In [5]:
for m in client.models.list():
    if "generateContent" in (m.supported_actions or []) and "flash" in m.name:
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3.1-flash-lite
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash


## 5. Run all postings
Results are cached in `data/extracted.json`, so re-running the notebook doesn't call the API again. Set `REFRESH = True` to force a fresh extraction (e.g. after changing the prompt).

In [ ]:
REFRESH = False

if CACHE_FILE.exists() and not REFRESH:
    records = json.loads(CACHE_FILE.read_text())
    print(f"Loaded {len(records)} cached extractions from {CACHE_FILE}")
else:
    records = []
    for f in sorted(POSTINGS_DIR.glob("*.txt")):
        print(f"Extracting {f.name}...")
        try:
            rec = extract(f.read_text()).model_dump()
            rec["file"] = f.name
            records.append(rec)
        except Exception as e:
            print(f"  FAILED on {f.name}: {e}")
        time.sleep(1)  # stay well under free-tier rate limits
    CACHE_FILE.parent.mkdir(exist_ok=True)
    CACHE_FILE.write_text(json.dumps(records, indent=2))
    print(f"Saved {len(records)} extractions to {CACHE_FILE}")

df = pd.DataFrame(records)
sources = pd.read_csv("sources.csv")[["file", "text_completeness"]]
df = df.merge(sources, on="file", how="left")
df[["file", "company", "job_title", "seniority", "work_mode", "min_years_experience"]]

Extracting posting_01.txt...
Extracting posting_02.txt...
  attempt 1 failed (ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp). Retrying in 5s...
  attempt 2 failed (ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp). Retrying in 10s...
  attempt 3 failed (ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp). Retrying in 20s...
  FAILED on posting_02.txt: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Extracting posting_03.txt...
  attempt 1 failed (ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp). Retrying in 5s...
  attempt 2 failed (ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp). Retrying in 10s...
  attempt 3 faile

## 6. Analysis
### 6.1 Clean skill names
The prompt asks for canonical names, but models aren't perfectly consistent, so we normalize casing and a few common variants before counting.

In [ ]:
ALIASES = {
    "ms excel": "Excel", "microsoft excel": "Excel", "advanced excel": "Excel",
    "powerbi": "Power BI", "microsoft power bi": "Power BI",
    "postgres": "PostgreSQL", "google bigquery": "BigQuery", "amazon redshift": "Redshift",
    "lookml": "Looker", "a/b tests": "A/B Testing", "ab testing": "A/B Testing",
    "experimentation": "A/B Testing", "statistical analysis": "Statistics",
    "github": "Git", "pandas": "Python", "numpy": "Python",
}

def clean_skills(skills):
    out = []
    for s in skills or []:
        s = s.strip()
        s = ALIASES.get(s.lower(), s)
        if s and s not in out:
            out.append(s)
    return out

# Use the most common capitalization for each skill (so "sql" and "SQL" merge)
all_raw = [s for col in ["required_skills", "nice_to_have_skills"] for lst in df[col] for s in clean_skills(lst)]
canonical = {}
for s, _ in pd.Series(all_raw).value_counts().items():
    canonical.setdefault(s.lower(), s)

for col in ["required_skills", "nice_to_have_skills"]:
    df[col] = df[col].apply(lambda lst: list(dict.fromkeys(canonical[s.lower()] for s in clean_skills(lst))))

n = len(df)
print(f"{n} postings, {len(canonical)} distinct skills after cleaning")

### 6.2 Most requested skills
Share of postings that list each skill as **required**.

In [ ]:
BLUE, ORANGE, INK, MUTED = "#2a78d6", "#eb6834", "#0b0b0b", "#52514e"

def style(ax, title):
    ax.set_title(title, loc="left", fontsize=13, color=INK, pad=12)
    for side in ["top", "right", "left"]:
        ax.spines[side].set_visible(False)
    ax.spines["bottom"].set_color("#d0d0cc")
    ax.tick_params(colors=MUTED, length=0)
    ax.grid(axis="x", color="#ececea", linewidth=0.8)
    ax.set_axisbelow(True)

req = df.explode("required_skills")["required_skills"].dropna().value_counts()
req_pct = (req / n * 100).round(0)

top = req_pct.head(15).sort_values()
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top.index, top.values, color=BLUE, height=0.6)
for y, v in enumerate(top.values):
    ax.text(v + 1, y, f"{v:.0f}%", va="center", fontsize=9, color=MUTED)
ax.set_xlim(0, 110)
ax.set_xlabel("% of postings listing it as required", color=MUTED)
style(ax, f"Most required skills across {n} Data Analyst postings")
plt.tight_layout()
plt.savefig(FIG_DIR / "top_required_skills.png", dpi=200)
plt.show()

pd.DataFrame({"postings": req, "% of postings": req_pct}).head(15)

### 6.3 Required vs. nice-to-have
Some tools show up mostly as a bonus rather than a requirement. That tells you what to learn first and what can wait.

In [ ]:
nice = df.explode("nice_to_have_skills")["nice_to_have_skills"].dropna().value_counts()
both = pd.DataFrame({"Required": req, "Nice to have": nice}).fillna(0).astype(int)
both["total"] = both.sum(axis=1)
both = both.sort_values("total", ascending=False).head(15).sort_values("total")

fig, ax = plt.subplots(figsize=(8, 6))
y = range(len(both))
ax.barh([i + 0.18 for i in y], both["Required"], height=0.34, color=BLUE, label="Required")
ax.barh([i - 0.18 for i in y], both["Nice to have"], height=0.34, color=ORANGE, label="Nice to have")
ax.set_yticks(list(y), both.index)
ax.set_xlabel("Number of postings", color=MUTED)
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
ax.legend(frameon=False, loc="lower right")
style(ax, "Required vs. nice-to-have mentions (top 15 skills)")
plt.tight_layout()
plt.savefig(FIG_DIR / "required_vs_nice.png", dpi=200)
plt.show()

### 6.4 Seniority, work mode and experience

In [ ]:
order_sen = ["intern", "entry", "mid", "senior", "lead"]
order_mode = ["remote", "hybrid", "onsite", "unspecified"]
sen = df["seniority"].value_counts().reindex(order_sen, fill_value=0)
mode = df["work_mode"].value_counts().reindex(order_mode, fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
for ax, s, title in [(axes[0], sen, "Seniority level"), (axes[1], mode, "Work mode")]:
    ax.barh(s.index[::-1], s.values[::-1], color=BLUE, height=0.6)
    for yy, v in enumerate(s.values[::-1]):
        ax.text(v + 0.15, yy, str(v), va="center", fontsize=9, color=MUTED)
    style(ax, title)
    ax.set_xlim(0, max(s.max(), 1) * 1.2)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
plt.tight_layout()
plt.savefig(FIG_DIR / "seniority_workmode.png", dpi=200)
plt.show()

yrs = df["min_years_experience"].dropna()
print(f"Minimum years of experience stated in {len(yrs)} of {n} postings: "
      f"median {yrs.median():.0f}, range {yrs.min():.0f}-{yrs.max():.0f}")

### 6.5 Salary
Only postings that list pay. Hourly rates are annualized at 2,080 hours (40 h × 52 weeks).

In [ ]:
pay = df[df["salary_min"].notna() | df["salary_max"].notna()].copy()
factor = pay["pay_period"].map({"hour": 2080}).fillna(1)
pay["annual_min"] = pay["salary_min"] * factor
pay["annual_max"] = pay["salary_max"] * factor
pay["midpoint"] = pay[["annual_min", "annual_max"]].mean(axis=1)

print(f"{len(pay)} of {n} postings list pay ({len(pay)/n:.0%}).")
if len(pay):
    print(f"Median midpoint: ${pay['midpoint'].median():,.0f} per year")
pay[["company", "job_title", "seniority", "salary_min", "salary_max", "pay_period", "midpoint"]].sort_values("midpoint")

## 7. Personal skill-gap analysis
Edit `MY_SKILLS` to match your own skills (use the same canonical names as the chart above). The cell shows how many of each posting's required skills you already have and which frequently requested skills are missing.

In [ ]:
MY_SKILLS = ["SQL", "Python", "Excel", "Tableau", "Statistics"]   # <- edit this list

mine = {s.lower() for s in MY_SKILLS}
df["required_matched"] = df["required_skills"].apply(lambda r: sum(s.lower() in mine for s in r))
df["required_total"] = df["required_skills"].apply(len)
df["match_pct"] = (df["required_matched"] / df["required_total"].where(df["required_total"] > 0) * 100).round(0)

print(f"Average share of required skills you have: {df['match_pct'].mean():.0f}%")
print(f"Postings where you meet at least 75% of required skills: {(df['match_pct'] >= 75).sum()} of {n}")

gaps = req_pct[[s.lower() not in mine for s in req_pct.index]].head(8)
print("\nMost-requested skills you don't list yet:")
for skill, pct in gaps.items():
    print(f"  {skill:<20} required in {pct:.0f}% of postings")

df[["company", "job_title", "seniority", "required_matched", "required_total", "match_pct"]].sort_values("match_pct", ascending=False)

## 8. Analyze your own posting
Paste any job posting into the box below and click **Analyze posting**. Gemini extracts it with the same schema, then you see:
- the extracted details
- which required skills you already have (based on `MY_SKILLS` in section 7)
- how common each skill is across the dataset

Click **Save to dataset** to add the posting to `postings/`, `sources.csv` and the cache. Then re-run from section 5 to update the charts (no `REFRESH` needed).

*Run sections 1–7 first. If the box doesn't appear, run `uv sync` in Terminal and restart the kernel.*

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

def analyze_posting(text: str) -> tuple[JobPosting, pd.DataFrame]:
    """Extract one posting and compare its required skills with MY_SKILLS and the dataset."""
    rec = extract(text)
    skills = list(dict.fromkeys(canonical.get(s.lower(), s) for s in clean_skills(rec.required_skills)))
    table = pd.DataFrame({
        "required skill": skills,
        "you have it": ["yes" if s.lower() in mine else "no" for s in skills],
        "% of dataset postings requiring it": [f"{req_pct.get(s, 0):.0f}%" for s in skills],
    })
    return rec, table

def save_posting(text: str, rec: JobPosting, url: str = "") -> str:
    """Write the posting to postings/, add a row to sources.csv and append it to the extraction cache."""
    for f in POSTINGS_DIR.glob("*.txt"):
        if f.read_text().strip() == text.strip():
            return f"Already in the dataset as {f.name}; nothing saved."
    nums = [int(f.stem.split("_")[-1]) for f in POSTINGS_DIR.glob("posting_*.txt") if f.stem.split("_")[-1].isdigit()]
    fname = f"posting_{max(nums, default=0) + 1:02d}.txt"
    (POSTINGS_DIR / fname).write_text(text.strip() + "\n")

    src = pd.read_csv("sources.csv")
    src.loc[len(src)] = [fname, rec.job_title, rec.company, "full posting (user added)", url or "not provided"]
    src.to_csv("sources.csv", index=False)

    cached = json.loads(CACHE_FILE.read_text()) if CACHE_FILE.exists() else []
    cached.append({**rec.model_dump(), "file": fname})
    CACHE_FILE.write_text(json.dumps(cached, indent=2))
    return f"Saved as {fname}. Re-run from section 5 to include it in the charts."

posting_box = widgets.Textarea(placeholder="Paste the full job posting here...",
                               layout=widgets.Layout(width="100%", height="260px"))
url_box = widgets.Text(placeholder="Job URL (optional, used in sources.csv)", layout=widgets.Layout(width="100%"))
analyze_btn = widgets.Button(description="Analyze posting", button_style="primary", icon="search")
save_btn = widgets.Button(description="Save to dataset", icon="save", disabled=True)
out = widgets.Output()
state = {}

def on_analyze(_):
    with out:
        clear_output()
        text = posting_box.value.strip()
        if len(text) < 100:
            print("Paste the full posting first (it looks empty or too short).")
            return
        print("Analyzing with Gemini...")
        try:
            rec, table = analyze_posting(text)
        except Exception as e:
            print(f"Extraction failed: {e}")
            return
        clear_output()
        state.update(text=text, rec=rec)
        pay = "not listed"
        if rec.salary_min or rec.salary_max:
            unit = "/hour" if rec.pay_period == "hour" else "/year"
            pay = f"${rec.salary_min or 0:,.0f} - ${rec.salary_max or 0:,.0f} {unit}"
        print(f"{rec.job_title} at {rec.company}")
        print(f"Seniority: {rec.seniority} | Work mode: {rec.work_mode} | "
              f"Min. experience: {rec.min_years_experience if rec.min_years_experience is not None else 'not stated'} yrs | Pay: {pay}")
        if len(table):
            have = (table["you have it"] == "yes").sum()
            print(f"\nYou have {have} of {len(table)} required skills ({have / len(table):.0%}).")
            missing = table.loc[table["you have it"] == "no", "required skill"].tolist()
            if missing:
                print("Missing:", ", ".join(missing))
            display(table)
        else:
            print("\nNo required skills were found in this posting.")
        if rec.nice_to_have_skills:
            print("Nice to have:", ", ".join(rec.nice_to_have_skills))
        save_btn.disabled = False

def on_save(_):
    with out:
        if "rec" in state:
            print("\n" + save_posting(state["text"], state["rec"], url_box.value.strip()))
            save_btn.disabled = True

analyze_btn.on_click(on_analyze)
save_btn.on_click(on_save)
display(widgets.VBox([posting_box, url_box, widgets.HBox([analyze_btn, save_btn]), out]))

## 9. Validation: spot-check the model
LLM extraction isn't guaranteed to be correct, so check a sample by hand. This cell prints three random postings next to what the model extracted. Read each one and note any mistakes (a missed skill, a nice-to-have marked as required, wrong seniority) in the table below. Change `SEED` to sample different postings.

In [ ]:
SEED = 7
for _, row in df.sample(3, random_state=SEED).iterrows():
    print("=" * 90)
    print((POSTINGS_DIR / row["file"]).read_text()[:1500])
    print("-" * 40, "EXTRACTED", "-" * 40)
    print(f"seniority={row['seniority']} | work_mode={row['work_mode']} | min_years={row['min_years_experience']}")
    print("required:     ", row["required_skills"])
    print("nice to have: ", row["nice_to_have_skills"])

**Spot-check results** (fill in after reading the output above):

| Posting | Errors found | Notes |
|---|---|---|
| | | |
| | | |
| | | |

## 10. Findings and limitations
**Key findings:** *(write 3–5 sentences from your charts: which skills dominate, how many postings are entry-level, what the salary data shows, what your skill gap is.)*

**Limitations:**
- **Small, hand-collected sample.** 20 postings collected in November 2026 don't represent the whole job market, and the set leans toward mid and senior roles.
- **Incomplete source text.** 14 of the 20 postings were reconstructed from search listings because the original pages had closed (see `text_completeness` in `sources.csv`). Shorter text means some skills may be missing.
- **Model judgment calls.** Required vs. nice-to-have and seniority are sometimes ambiguous in the postings themselves, and the model's interpretation can differ from a human's. Section 9 measures how often.
- **Skill-name normalization** is partly prompt-driven and partly rule-based (`ALIASES`). Unusual variants can still slip through as separate skills.
- **Salary data is sparse** because many postings don't list pay.